In [1]:
# 让 notebook 无论从哪个目录启动都能 import retailinsight
import sys
from pathlib import Path

_candidates = [Path.cwd(), *Path.cwd().parents]
_root = next(p for p in _candidates if (p / "retailinsight" / "__init__.py").exists())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print("项目根目录:", _root)


项目根目录: /Users/lizhipeng/个人成长/职业发展/求职面试/项目/RetailInsight


# 02 · RFM 与会员画像
> 目标：把会员切成 8 个可运营的客群，并给出每个客群的画像特征。

In [2]:
from retailinsight.io.loader import read_parquet
from retailinsight.settings import MART_PROFILE, MART_RFM

rfm = read_parquet(MART_RFM)
profile = read_parquet(MART_PROFILE)
print(f"mart_rfm: {rfm.shape}  mart_profile: {profile.shape}")


mart_rfm: (35650, 10)  mart_profile: (36000, 51)


## 2.1 RFM 打分口径
> 五分位打分 1~5，≥4 记「高」，再按 2×2×2 映射成 8 类客群。

In [3]:

print(rfm[["recency_days", "frequency", "monetary", "r_score", "f_score", "m_score",
           "rfm_segment"]].describe().round(2))


       recency_days  frequency  monetary   r_score   f_score   m_score
count      35650.00   35650.00  35650.00  35650.00  35650.00  35650.00
mean         152.81       8.35   1199.86      3.00      3.00      3.00
std          205.70       8.79   1349.62      1.41      1.41      1.41
min            1.00       1.00   -450.92      1.00      1.00      1.00
25%           20.00       1.00    193.08      2.00      2.00      2.00
50%           55.00       5.00    728.24      3.00      3.00      3.00
75%          197.00      12.00   1756.59      4.00      4.00      4.00
max          943.00      57.00  10570.45      5.00      5.00      5.00


## 2.2 八类客群分布与 GMV 贡献

In [4]:
summary = (rfm.groupby("rfm_segment")
             .agg(members=("member_id", "count"), gmv=("monetary", "sum"),
                  avg_recency=("recency_days", "mean"), avg_frequency=("frequency", "mean"),
                  avg_monetary=("monetary", "mean"))
             .round(2))
summary["member_share"] = (summary["members"] / len(rfm)).round(4)
summary["gmv_share"] = (summary["gmv"] / summary["gmv"].sum()).round(4)
summary.sort_values("gmv", ascending=False)


,members,gmv,avg_recency,avg_frequency,avg_monetary,member_share,gmv_share
rfm_segment,,,,,,,
重要价值客户,6531,1.818520e+07,17.68,19.50,2784.439941,0.1832,0.4251
重要保持客户,6507,1.509587e+07,129.32,15.60,2319.939941,0.1825,0.3529
流失风险客户,13485,4.171924e+06,305.39,2.28,309.380005,0.3783,0.0975
潜力客户,6683,2.534407e+06,16.19,2.79,379.230011,0.1875,0.0592
重要挽留客户,736,9.894338e+05,173.88,6.66,1344.339966,0.0206,0.0231
重要发展客户,486,6.825229e+05,17.01,6.40,1404.369995,0.0136,0.0160
一般保持客户,662,6.122388e+05,178.61,9.25,924.830017,0.0186,0.0143
一般价值客户,560,5.033641e+05,17.52,9.69,898.859985,0.0157,0.0118


## 2.3 客群特征对比（画像宽表）
> 画像宽表已 join 流失概率（第 20 个 P0 标签），`churn_covered` 标注是否被覆盖。

In [5]:
cols = ["avg_basket", "monthly_amount", "yearly_freq", "items_per_txn",
        "category_entropy", "discount_sensitivity", "weekend_ratio"]
cmp = profile.groupby("rfm_segment")[cols].mean().round(3)
cmp.join(summary[["members", "gmv_share"]])


,avg_basket,monthly_amount,yearly_freq,items_per_txn,category_entropy,discount_sensitivity,weekend_ratio,members,gmv_share
rfm_segment,,,,,,,,,
一般价值客户,93.396,29.007999,13.236,5.326,3.518,0.062,0.290,560,0.0118
一般保持客户,100.892,29.846001,7.331,6.436,3.638,0.060,0.295,662,0.0143
流失风险客户,137.066,9.984000,3.162,5.821,2.262,0.040,0.291,13485,0.0975
潜力客户,139.008,12.238000,11.485,6.851,2.668,0.066,0.312,6683,0.0592
重要价值客户,146.512,89.859001,12.127,7.350,3.907,0.048,0.298,6531,0.4251
重要保持客户,149.572,74.869003,8.679,7.796,3.798,0.050,0.301,6507,0.3529
重要发展客户,223.166,45.320999,11.548,9.954,3.592,0.039,0.312,486,0.0160
重要挽留客户,203.974,43.383999,5.870,9.170,3.464,0.036,0.319,736,0.0231


## 2.4 流失概率覆盖率说明
> 流失得分只覆盖末期 cohort 观察窗内的活跃会员，其余为 NaN（保留而非填均值）。

In [6]:
if "churn_prob" in profile.columns:
    cov = profile["churn_covered"].mean()
    print(f"churn_prob 覆盖率: {cov:.1%}（未覆盖 {int((~profile['churn_covered']).sum()):,} 人保留 NaN）")
    print(profile.groupby("rfm_segment")["churn_prob"].mean().round(4).sort_values(ascending=False))
else:
    print("未生成 churn_prob，请先运行 python -m retailinsight.cli run --stage profile")


churn_prob 覆盖率: 63.3%（未覆盖 13,223 人保留 NaN）
rfm_segment
流失风险客户    0.6713
潜力客户      0.4557
重要挽留客户    0.4356
一般保持客户    0.4133
重要发展客户    0.3956
重要保持客户    0.3451
一般价值客户    0.2916
重要价值客户    0.2376
Name: churn_prob, dtype: float64


## 2.5 分群雷达图

In [7]:
from retailinsight.viz import charts

print(charts.segment_radar(profile, by="rfm_segment"))


/Users/lizhipeng/个人成长/职业发展/求职面试/项目/RetailInsight/reports/figures/11_segment_radar.png
